# 6 · plot — structure accuracy against MSA depth

Draws [`6_make_msa_depth_data.ipynb`](6_make_msa_depth_data.ipynb) as one panel per metric:
mean accuracy against MSA depth bin, on natural FoldBench monomers.

Edit `ARMS` to choose which predictors appear and what they are called; the dataset holds every
arm helico exp14 ran.

In [ ]:
# Run from anywhere: figlib lives next to this notebook.
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "figlib.py").exists() else Path("experiments/exp250_evals_exploration_notebook/figures")
sys.path.insert(0, str(HERE.resolve()))
import figlib

In [ ]:
DATASET = "6_msa_depth"
DPI = 300
# arm -> label. The same six predictors figure 3 draws, so a reader carries the bars over: two
# that see an alignment (Protenix-v2 MSA, and ESMFold2 through its language model), two that see
# one sequence, plus the floor and ceiling on contact conditioning.
ARMS = {
    "oracle": "Helico + true contacts",
    "mf_L_363k": "Helico + MarinFold contacts",
    "off": "Helico, no contacts",
    "protenix_v2_single_seq": "Protenix-v2 SS",
    "esmfold2": "ESMFold2",
    "protenix_v2_msa": "Protenix-v2 MSA",
}
HIGHLIGHT = "mf_L_363k"
METRIC_LABELS = {"gdt_ts": "GDT-TS", "lddt": "lDDT"}
metadata = figlib.describe(DATASET)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

figlib.figure_style(DPI)
summary = pd.read_csv(figlib.require(DATASET, "summary.csv") / "summary.csv")
occupancy = figlib.load_metadata(DATASET)["occupancy"]
ACCENT, NEUTRAL = "#C44E52", "#7A8DA6"
#: Width of a panel that will sit two-up in a 6.5 in text column, as in figures 2 and 3.
PANEL_WIDTH = 3.25
ORDER = list(occupancy)


def panel(frame, metric, name):
    """One line per arm across the depth bins, with its bootstrap interval."""
    figure, axis = plt.subplots(figsize=(PANEL_WIDTH, 2.5), layout="constrained")
    positions = range(len(ORDER))
    for arm, label in ARMS.items():
        line = frame[frame.arm == arm].set_index("depth_bin").reindex(ORDER)
        accent = arm == HIGHLIGHT
        axis.errorbar(positions, line.value,
                      yerr=[line.value - line.ci_low, line.ci_high - line.value],
                      marker="o", markersize=3.5, lw=1.8 if accent else 1.1,
                      color=ACCENT if accent else NEUTRAL,
                      ecolor=ACCENT if accent else NEUTRAL, elinewidth=0.9, capsize=2,
                      zorder=3 if accent else 2, label=label)
        # Every line is the same two colours, so each is named at its right end instead of in a
        # legend that would cover the panel.
        axis.annotate(label, (positions[-1], line.value.iloc[-1]), xytext=(4, 0),
                      textcoords="offset points", va="center", fontsize=6.5,
                      color=ACCENT if accent else "0.35")
    axis.set(xlabel="MSA depth (sequences)", ylabel=METRIC_LABELS.get(metric, metric),
             xticks=list(positions), ylim=(0, 1.02))
    axis.set_xticklabels([f"{label}\n(n={occupancy[label]})" for label in ORDER], fontsize=7)
    axis.grid(axis="y", alpha=0.25, lw=0.6)
    axis.set_axisbelow(True)
    figure.text(0.005, 0.005, "FoldBench natural monomers", ha="left", va="bottom", fontsize=8,
                color="0.35")
    figlib.save_figure(figure, name, DPI)
    plt.show()


for metric in summary.metric.unique():
    frame = summary[summary.metric == metric]
    print(f"--- {METRIC_LABELS.get(metric, metric)} ---")
    print(frame[frame.arm.isin(ARMS)].pivot(index="arm", columns="depth_bin", values="value")
          .reindex(list(ARMS))[ORDER].to_string(float_format=lambda v: f"{v:.3f}"))
    panel(frame, metric, f"msa_depth_{metric}")